# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [2]:
import torch
import torch.nn as nn
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):#d_model就是维度，num_heads就是多头的头数
        self.num_heads = num_heads
        self.d_k = d_model // num_heads#这个就是多头中一个头的最后一个的维度

        #下面是四个线性
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)

        # Initialize W_q, W_k, W_v, W_o

    def forward(self, Q, K, V):
        #首先，一个Q的维度应该是(batch,len,d_model),即经过embedding后的一堆batch的向量
        #对于多头，我们需要计算出这三个维度，然后每一个头的维度就是d_model/num_head,
        B,S_q,_ = Q.size()

        S_k = K.shape[1]
        #先经过权重矩阵，然后吧后面的拆了，view后的维度是(batch,len,nums_head,d_k),然后调换一下nums-head和
        q = self.W_q(Q).view(B,S_q,self.num_heads,self.d_k).transpose(1,2)
        k = self.W_k(K).view(B,S_k,self.num_heads,self.d_k).transpose(1,2)
        v = self.W_v(V).view(B,S_k,self.num_heads,self.d_k).transpose(1,2)

        #然后在后面两个维度上计算attention
        score = torch.matmul(q,k.transpose(-1,-2))/math.sqrt(self.d_k)

        weight = torch.softmax(score,dim = -1)

        atten = torch.matmul(weight,v)

        #最后记得变回来维度
        output = atten.transpose(1,2).contiguous().view(B,S_q,-1)
        return self.W_o(output)

        

        
         # Implement multi-head attention

In [7]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [8]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (1.8ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.5ms)
  ✅ [3/6] Numerical correctness vs reference (1.7ms)
  ✅ [4/6] Gradient flow (1.6ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.3ms)
  ✅ [6/6] Different heads give different outputs (0.7ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (6.6ms total)
  Progress saved. Run status() to see your dashboard.



第一阶段：初始张量输入假设当前处理的批次大小为 $B$，序列长度为 $L$，模型的隐状态维度为 $d_{model}$（即 hidden_dim）。在进入多头注意力模块之前，模型提供的初始 Query ($Q$)、Key ($K$)、Value ($V$) 张量的形状通常均为：(B, L, d_model)

第二阶段：线性投影与多头切分（对应公式：$Q W_i^Q,\; K W_i^K,\; V W_i^V$）公式中表达的对每个头进行独立的线性映射，在实际的大模型底层工程中，为了最大化硬件并行效率，并不会使用 $h$ 个独立的较小线性层，而是通过一次完整的线性变换加张量重塑来实现等价操作。设定多头参数：假设设定的注意力头数为 $h$，则每个头的特征维度大小定义为 $d_k = d_{model} / h$。全局线性映射：使用三个独立的权重矩阵 $W^Q, W^K, W^V$（形状均为 (d_model, d_model)），分别对原始的 $Q, K, V$ 进行矩阵乘法运算。映射后的输出张量形状依然是 (B, L, d_model)。特征空间切分（Reshape & Transpose）：将张量在最后一个维度上切分为 $h$ 个大小为 $d_k$ 的子空间，调用 view 操作将形状重塑为 (B, L, h, d_k)。为了满足后续计算注意力分数的矩阵乘法要求（将 $h$ 提取至批次维度），必须执行转置操作 transpose(1, 2)。此时，$Q, K, V$ 的最终形状严格变换为：(B, h, L, d_k)

第三阶段：并行执行注意力计算（对应公式：$\text{head}_i = \text{Attention}(\dots)$）由于 $Q, K, V$ 均已被重塑为四维张量，多头之间的计算相互完全独立。通过调用上一题实现的 scaled_dot_product_attention 算子，框架会利用底层的高维矩阵乘法广播机制，对所有的头进行并发计算：计算注意力分数：Q @ K.transpose(-2, -1)，形状推导为 (B, h, L, d_k) @ (B, h, d_k, L)，输出形状为 (B, h, L, L)。应用 Softmax 归一化：在 dim=-1 上执行概率映射。聚合 Value 特征：将概率矩阵与 $V$ 相乘，即 (B, h, L, L) @ (B, h, L, d_k)。该阶段完成后的输出张量形状保持为：(B, h, L, d_k)

第四阶段：多头特征的拼接（对应公式：$\text{Concat}(\text{head}_1, \dots, \text{head}_h)$）在数学定义上，拼接（Concat）意味着将各个子空间独立提取的特征还原为完整的高维表示。在张量操作层面，需要执行前面转置操作的逆操作：反向转置：调用 transpose(1, 2)，将形状从 (B, h, L, d_k) 恢复为 (B, L, h, d_k)。内存连续化与视图重组：由于转置破坏了张量在底层物理内存中的连续性，必须先调用 .contiguous() 重新分配连续内存，随后使用 .view(B, L, d_model)。此时，多头维度的特征被物理地拼接在了一起，输出张量形状还原为：(B, L, d_model)

第五阶段：最终输出投影（对应公式：乘以 $W^O$）拼接后的特征虽然在维度上恢复到了 $d_{model}$，但各个头的特征在信息流上依然是严格隔离的。为了实现不同子空间信息的全局融合，需要施加最后一次仿射变换。将拼接后的张量送入最后一个输出线性层（权重矩阵 $W^O$ 的形状为 (d_model, d_model)）。经过 Concat_Matrix @ W^O.T 操作后，模块的最终输出结果为：(B, L, d_model)

综上所述，多头注意力模块接收三个形状为 (B, L, d_model) 的输入向量，在内部完成了特征的降维、并行交互与升维融合，最终输出一个形状完全一致的 (B, L, d_model) 张量，供下一层的残差连接（Residual Connection）与层归一化（LayerNorm）调用。

在pytorch中，数据是一直连续存储的，比如1 2 3 4 5 6 7 8

如果这个是一个2*4的，那么就是

1 2 3 4

5 6 7 8

但是pytorch是用一个stride来去构建的，即3 2，意思就是，横着走三步，竖着走两步，这样就可以了，那么我们在操作的时候就可以只改stride就行了

然后view也是，一种这样的操作，通过修改stride

那么最后什么是contiguous？就是如果我把这个东西给改了（比如乱七八糟改完后），他的读取顺序是  1 5 2 6 3 7 4 8，但是，原来内存里面是1 2 3 4 5 这样，读1和5的时候内存里面的还有2 3 4，这就是不连续，如果我们要连续，就直接使用comtigous，相当于在内存新开辟一块地方，让 1 5 2 6 3 7 4 8 这样存起来就行了，这样读取就是连续的了